In [1]:
# 生成大客户的台账脚本
%run merge_requirement_to_template.py

CONFIG_PATH: E:\python_source\KAST\merge_requirement_to_template.ini
运行时间: 2026-04-15 09:06:19
模板文件: E:\python_source\需求导入\更新导入模板（20260408）.xlsx
输入文件:
 - E:\python_source\需求导入\业务要求(20260415-记录数19).xlsx
 - E:\python_source\需求导入\业务要求(20260415-记录数9).xlsx
输出文件: E:\python_source\需求导入\更新导入模板（20260408）_20260415_090618.xlsx
日志文件: E:\python_source\KAST\merge_requirement_to_template_20260415_090619.log
导入原始记录数: 28
新增记录数(追加到模板末尾): 20
去重后总记录数: 94
重复记录数(按工单编号/ITM编号): 8
重复工单编号/ITM编号值:
 - BJ260401R128
 - HA260414R020
 - DL260408R204
 - DL260326R165
 - YN260414R073
 - GZ260413R179
 - NM260402R047
 - YN260326R085


In [10]:
%run generate_dualweekly_workreport.py

compose_map: {"协调情况": "\"该需求【\"+工单编号/ITM编号+\"】\n\"+处理意见"}
运行时间: 2026-03-30 19:11:00
配置文件: E:\python_source\KAST\workreport-JK.ini
模板文件: E:\python_source\需求导入\双周报需求模板.docx
使用表格索引: 0
输入文件:
 - E:\python_source\需求导入\需求台账模板（20260330）.xlsx
输出文件: E:\python_source\需求导入\双周报需求模板_20260330_6.docx
日志文件: E:\python_source\KAST\generate_dualweekly_workreport_20260330_191100.log
模板原有表格数据行数: 1
本次新增记录数: 2
输出表格总行数(含表头): 4
定位说明: Word 标准 docx 无法在打开时自动将光标跳到表格；请使用「查找」->「转到」->「书签」-> 名称「NewImportedData」跳转到首条新增行。


In [3]:
# 将word转换为markdown格式的工具
from docx import Document
docx_path = r'E:\04-大客户服务项目配置库\98-自我完成\2026年一季度个人工作总结-张峰.docx'
md_path =  r'E:\04-大客户服务项目配置库\98-自我完成\P_2026年一季度个人工作总结-张峰.md'
def docx_to_markdown(docx_path, md_path):
    doc = Document(docx_path)
    with open(md_path, 'w', encoding='utf-8') as f:
        for para in doc.paragraphs:
            f.write(para.text + '\\\\n\\\\n')

# 使用示例
docx_to_markdown(docx_path, md_path)

In [6]:
# 将word转换为markdown格式的工具
from docx import Document
docx_path = r'E:\04-大客户服务项目配置库\98-自我完成\2026年一季度个人工作总结-张峰.docx'
md_path =  r'E:\04-大客户服务项目配置库\98-自我完成\P_2026年一季度个人工作总结-张峰.md'


def get_heading_level(paragraph):
    """
    获取段落的标题级别
    :param paragraph: Word 文档中的段落对象
    :return: 标题级别（0 表示不是标题，1-6 分别对应 Markdown 的 # 到 ######）
    """
    if paragraph.style.name.startswith('Heading'):
        try:
            level = int(paragraph.style.name.split(' ')[1])
            return min(level, 6)  # Markdown 标题最多 6 级
        except (IndexError, ValueError):
            return 0
    return 0

def docx_table_to_markdown(table):
    """
    将 Word 表格转换为 Markdown 表格格式
    :param table: Word 文档中的表格对象
    :return: Markdown 格式的表格字符串
    """
    markdown_rows = []
    
    # 提取表格的行和列数据
    for row in table.rows:
        row_cells = [cell.text.strip() for cell in row.cells]
        markdown_rows.append("| " + " | ".join(row_cells) + " |")
    
    # 生成 Markdown 表格的分隔行（根据列数动态生成）
    if markdown_rows:
        num_columns = len(markdown_rows[0].split('|')[1:-1])  # 排除首尾的 |
        separator_row = "| " + " | ".join(["---"] * num_columns) + " |"
        return "\\n".join([markdown_rows[0], separator_row] + markdown_rows[1:])
    return ""

def docx_to_markdown_with_outline_and_tables(docx_path, md_path):
    """
    将 Word 文档转换为 Markdown 格式，保留大纲结构并处理表格
    :param docx_path: Word 文档路径
    :param md_path: 输出的 Markdown 文件路径
    """
    doc = Document(docx_path)
    with open(md_path, 'w', encoding='utf-8') as f:
        for element in doc.element.body:
            # 处理段落（包括标题和普通段落）
            if element.tag.endswith('p'):
                para = docx.text.paragraph.Paragraph(element, doc)
                heading_level = get_heading_level(para)
                if heading_level > 0:
                    # 写入 Markdown 标题
                    f.write('#' * heading_level + ' ' + para.text + '\\n\\n')
                elif para.text.strip():  # 忽略空段落
                    # 写入普通段落
                    f.write(para.text + '\\n\\n')
            # 处理表格
            elif element.tag.endswith('tbl'):
                # 由于 python-docx 的表格处理方式，这里需要重新解析表格
                # 更简单的方式是直接遍历 doc.tables
                pass  # 此处简化处理，实际需优化（见下方改进）
        
        # 改进的表格处理方式（直接遍历 doc.tables）
        # 由于上述 element 遍历方式在处理表格时较为复杂，改用直接遍历 doc.tables
        # 需要重新读取文档（或优化逻辑），此处采用重新读取的方式简化代码
        doc = Document(docx_path)  # 重新读取文档
        markdown_content = []
        for para in doc.paragraphs:
            heading_level = get_heading_level(para)
            if heading_level > 0:
                markdown_content.append('#' * heading_level + ' ' + para.text + '\\n\\n')
            elif para.text.strip():
                markdown_content.append(para.text + '\\n\\n')
        
        for table in doc.tables:
            markdown_table = docx_table_to_markdown(table)
            if markdown_table:
                markdown_content.append(markdown_table + "\\n")
        
        # 将所有内容写入文件
        f.write(''.join(markdown_content))

# 更简洁的实现方式（避免重复读取文档）
def docx_to_markdown_optimized(docx_path, md_path):
    """
    优化后的实现，避免重复读取文档
    :param docx_path: Word 文档路径
    :param md_path: 输出的 Markdown 文件路径
    """
    doc = Document(docx_path)
    markdown_content = []
    
    # 先处理所有段落
    for para in doc.paragraphs:
        heading_level = get_heading_level(para)
        if heading_level > 0:
            markdown_content.append('#' * heading_level + ' ' + para.text + '\\n\\n')
        elif para.text.strip():
            markdown_content.append(para.text + '\\n\\n')
    
    # 再处理所有表格
    for table in doc.tables:
        markdown_table = docx_table_to_markdown(table)
        if markdown_table:
            markdown_content.append(markdown_table + "\\n")
    
    # 将所有内容写入文件
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(''.join(markdown_content))

# 使用示例
docx_to_markdown_optimized(docx_path, md_path)

In [9]:
%pip install -i https://pypi.tuna.tsinghua.edu.cn/simple python-pptx

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [12]:
# 生成PPT的脚本
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor

def add_footer_bad(slide, page_num, date_text="2026年4月"):
    """在幻灯片底部添加页脚信息，按简单占位风格实现；若有模板字段可进一步对齐"""
    width = slide.part.chart.chart_space.get_parent().width  # 尝试获取宽度
    # 统一底部文本框：左中右布局
    left = slide.shapes.add_textbox(Inches(0.25), Inches(6.9), Inches(1.2), Inches(0.4))
    left_tf = left.text_frame
    left_tf.text = "LOGO"  # 左侧 LOGO 占位
    left_tf.paragraphs[0].alignment = PP_ALIGN.LEFT

    mid = slide.shapes.add_textbox(Inches(2.0), Inches(6.9), Inches(4.0), Inches(0.4))
    mid_tf = mid.text_frame
    mid_tf.text = f"Page {page_num}"
    mid_tf.paragraphs[0].alignment = PP_ALIGN.CENTER

    right = slide.shapes.add_textbox(Inches(6.0), Inches(6.9), Inches(2.5), Inches(0.4))
    right_tf = right.text_frame
    right_tf.text = date_text
    right_tf.paragraphs[0].alignment = PP_ALIGN.RIGHT

    # 字体美化（可选）
    for tf in [left_tf, mid_tf, right_tf]:
        for p in tf.paragraphs:
            for run in p.runs:
                run.font.size = Pt(10)
                run.font.color.rgb = RGBColor(0x00, 0x00, 0x00)

def add_footer(slide, prs, page_num, date_text="2026年4月"):
    """在幻灯片底部添加页脚信息，使用固定位置，不依赖图表对象"""
    slide_width = prs.slide_width  # 测量单位是EMU
    # 将尺寸转换为英寸，常用的 1 英寸 = 914400 EMU
    left_x = Inches(0.25)
    right_x = Inches(9.25)  # 根据你的幻灯片宽度调整，一般 13.33" 宽度
    top_y = Inches(6.9)
    box_w = Inches(1.8)
    box_h = Inches(0.4)

    # 左侧 LOGO 占位文本框（替代实际 LOGO）
    left = slide.shapes.add_textbox(left_x, top_y, box_w, box_h)
    left_tf = left.text_frame
    left_tf.text = "LOGO"

    # 中间页码
    mid = slide.shapes.add_textbox(Inches(4.0), top_y, Inches(4.0), box_h)
    mid_tf = mid.text_frame
    mid_tf.text = f"Page {page_num}"
    mid_para = mid_tf.paragraphs[0]
    mid_para.alignment = PP_ALIGN.CENTER

    # 右侧日期
    right = slide.shapes.add_textbox(Inches(9.0), top_y, box_w, box_h)
    right_tf = right.text_frame
    right_tf.text = date_text
    right_para = right_tf.paragraphs[0]
    right_para.alignment = PP_ALIGN.RIGHT

    # 统一字体
    for tf in [left_tf, mid_tf, right_tf]:
        for p in tf.paragraphs:
            for run in p.runs:
                run.font.size = Pt(10)
                run.font.color.rgb = RGBColor(0x00, 0x00, 0x00)


def add_bullets(text_frame, items, level=0):
    """辅助函数：向文本框添加多级项目符号"""
    p = text_frame.add_paragraph()
    p.level = level
    p.text = items[0]
    p.font.size = Pt(12)
    for it in items[1:]:
        p = text_frame.add_paragraph()
        p.text = it
        p.level = level + 1
        p.font.size = Pt(12)

def main():
    prs = Presentation()

    # 页眉/页脚模板设定：统一日期
    date_text = "2026年4月"  # 这里可改为 "2026-04-01" 或 "Q1 2026" 等

    # 第1页：标题页
    slide1 = prs.slides.add_slide(prs.slide_layouts[5])  # 使用空白布局，方便定制
    title_box = slide1.shapes.add_textbox(Inches(1.0), Inches(1.5), Inches(8.0), Inches(0.8))
    tf = title_box.text_frame
    tf.text = "2026年一季度个人工作总结"
    tf.paragraphs[0].font.size = Pt(32)
    tf.paragraphs[0].font.bold = True

    # 次要信息
    sub1 = slide1.shapes.add_textbox(Inches(1.0), Inches(2.6), Inches(8.0), Inches(0.5))
    tf2 = sub1.text_frame
    tf2.text = "张峰"
    tf2.paragraphs[0].font.size = Pt(18)

    sub2 = slide1.shapes.add_textbox(Inches(1.0), Inches(3.2), Inches(8.0), Inches(0.5))
    tf3 = sub2.text_frame
    tf3.text = "时间区间：2026年1月—3月"
    tf3.paragraphs[0].font.size = Pt(14)

    add_footer(slide1, 1, date_text)

    # 第2页：年度OKR 概览
    slide2 = prs.slides.add_slide(prs.slide_layouts[5])
    title2 = slide2.shapes.add_textbox(Inches(0.8), Inches(0.8), Inches(9.0), Inches(0.7))
    t2 = title2.text_frame
    t2.text = "2026年年度工作OKR（概览）"
    t2.paragraphs[0].font.size = Pt(20)
    t2.paragraphs[0].font.bold = True

    # 内容要点
    body2 = slide2.shapes.add_textbox(Inches(0.8), Inches(1.7), Inches(9.0), Inches(4.0))
    tb2 = body2.text_frame
    tb2.word_wrap = True
    items = [
        "目标1：完成大客户视图系统建设",
        "  - 关键结果：完成需求编写和原型设计；完成时间：2026年7月",
        "目标2：加强个人学习与能力提升",
        "  - 关键结果：学习现金管理、司库系统、资金监管系统；完成时间：2026年12月",
        "目标3：重点大客户支持服务",
        "  - 关键结果：全年做好雅江、一汽大客户的支持；完成时间：2026年12月",
        "核心要点：三大核心目标贯穿，驱动分行服务与大客户支持",
    ]
    add_bullets(tb2, items)

    add_footer(slide2, 2, date_text)

    # 第3页：一季度工作完成情况（分主线梳理）
    slide3 = prs.slides.add_slide(prs.slide_layouts[5])
    title3 = slide3.shapes.add_textbox(Inches(0.8), Inches(0.8), Inches(9.0), Inches(0.7))
    t3 = title3.text_frame
    t3.text = "一季度工作完成情况（分主线梳理）"
    t3.paragraphs[0].font.size = Pt(20)
    t3.paragraphs[0].font.bold = True

    body3 = slide3.shapes.add_textbox(Inches(0.8), Inches(1.6), Inches(9.0), Inches(3.5))
    tb3 = body3.text_frame

    sections = [
        ("分行大客户工作支持",
         [
             " 常规支持：江苏、浙江、吉林、云南、山西分行",
             " 专项任务要点：一汽集团司库系统、吉林烟草、云南财政模式、江苏海力士、浙江淘宝卡、山西鹏飞等"
         ]),
        ("雅江集团营销工作支持",
         [
             " 日常服务：双周报、投标跟进",
             " 项目支持：大模型部署及司库优化需求沟通"
         ]),
        ("需求跟进组工作",
         [
             " 机制建设：年度工作方案制定",
             " 过程监控：双周报、排期跟踪",
             " 分析改进：30天以上排期根因分析与流程优化"
         ]),
        ("大客户视图原型工作",
         [
             " 设计与评审：首页/需求跟进视图/工作台的原型与评审",
             " 需求优化：35处优化"
         ]),
    ]

    for section, bullets in sections:
        add_section = tb3.add_paragraph()
        add_section.text = section
        add_section.font.size = Pt(13)
        add_section.font.bold = True
        for b in bullets:
            p = tb3.add_paragraph()
            p.text = b
            p.font.size = Pt(12)
            p.level = 1

    add_footer(slide3, 3, date_text)

    # 第4页：工作反思与成长、后续计划
    slide4 = prs.slides.add_slide(prs.slide_layouts[5])
    title4 = slide4.shapes.add_textbox(Inches(0.8), Inches(0.8), Inches(9.0), Inches(0.7))
    t4 = title4.text_frame
    t4.text = "工作反思与成长、后续计划"
    t4.paragraphs[0].font.size = Pt(20)
    t4.paragraphs[0].font.bold = True

    body4 = slide4.shapes.add_textbox(Inches(0.8), Inches(1.6), Inches(9.0), Inches(3.5))
    tb4 = body4.text_frame
    bullets4 = [
        ("个人能力提升", [
            " 深入了解ITM与RAT联动",
            " 熟练使用 Axure 进行原型设计"
        ]),
        ("团队协作致谢", [
            " 感谢同事与领导的支持"
        ]),
        ("后续提升计划", [
            " 深化需求管理全流程与多元场景",
            " 拓展对公大客户信息管理与群组关系理解"
        ])
    ]
    for title, items in bullets4:
        p = tb4.add_paragraph()
        p.text = title
        p.font.size = Pt(13)
        p.font.bold = True
        for it in items:
            p = tb4.add_paragraph()
            p.text = it
            p.font.size = Pt(12)
            p.level = 1

    add_footer(slide4, 4, date_text)

    # 保存
    output_filename = "2026_Q1_张峰_个人工作总结.pptx"
    prs.save(output_filename)
    print(f"PPT 已生成: {output_filename}")

if __name__ == "__main__":
    main()


AttributeError: 'int' object has no attribute 'slide_width'

In [1]:
%run generate_dualweekly_workreport.py

compose_map: {"协调情况": "\"该需求【\"+工单编号/ITM编号+\"】\n\"+处理意见"}
运行时间: 2026-04-10 10:02:19
配置文件: E:\python_source\KAST\workreport-JK.ini
模板文件: E:\python_source\需求导入\双周报需求模板.docx
使用表格索引: 0
输入文件:
 - E:\python_source\需求导入\需求台账模板（20260410）.xlsx
输出文件: E:\python_source\需求导入\双周报需求模板_20260410.docx
日志文件: E:\python_source\KAST\generate_dualweekly_workreport_20260410_100219.log
模板原有表格数据行数: 1
本次新增记录数: 4
输出表格总行数(含表头): 6
定位说明: Word 标准 docx 无法在打开时自动将光标跳到表格；请使用「查找」->「转到」->「书签」-> 名称「NewImportedData」跳转到首条新增行。
